# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use **Logistic Regression, a shallow Decision Tree, and a constrained Random Forest** on the same feature vector. Logistic Regression gives a simple linear reference, the Decision Tree gives an interpretable nonlinear model, and Random Forest can capture interactions without relying on a single tree. I will not choose the most complex model automatically: the preferred model is the one that improves the Week-4 baseline on the holdout, especially on `Precision@50`, while remaining reasonable for this decision-support task.

In [12]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

# Locate the repository in Colab or local Jupyter.
candidates = [Path("/content/Flyrank1"), Path.cwd()]
REPO_ROOT = next((p for p in candidates if (p / "data").exists()), None)
if REPO_ROOT is None:
    matches = list(Path("/content").glob("**/data/raw/content_refresh_anonymized.csv"))
    if matches:
        REPO_ROOT = matches[0].parents[2]
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the FlyRank repo. Run this notebook from the repo or clone it in Colab.")

PROCESSED = REPO_ROOT / "data" / "processed"
FEATURE_PATH = PROCESSED / "refresh_feature_vector.csv"
BASELINE_PATH = PROCESSED / "baseline_refresh_queue.csv"

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Missing {FEATURE_PATH}. Run the repository feature-preparation pipeline first."
    )

frame = pd.read_csv(FEATURE_PATH)
print("Repository:", REPO_ROOT)
print("Rows:", len(frame))

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "is_declining_label"

missing = [c for c in FEATURES + [TARGET, "client_id", "content_id"] if c not in frame.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Clean the same way each model sees the feature vector.
for col in NUMERIC_FEATURES:
    frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for col in CATEGORICAL_FEATURES:
    frame[col] = frame[col].fillna("unknown").astype(str)
frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").astype(int)

X = frame[FEATURES].copy()
y = frame[TARGET].copy()
print("Feature count:", len(FEATURES))
print("Declining rate:", f"{y.mean():.3%}")


Repository: /content/Flyrank1
Rows: 30000
Feature count: 26
Declining rate: 54.207%


## 2. Split design

The split is **client-level**, not row-level. A random 20% of unique clients is held out, so pages from the same client cannot appear in both training and test. This is the honest choice because client-level similarity could otherwise make a row split look better than it generalizes to unseen clients. The split is deterministic with seed 42.

In [13]:
# Deterministic client-level holdout.
clients = frame["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()

if len(unique_clients) < 5:
    raise ValueError("Not enough unique clients for a client-level holdout.")

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.flatnonzero(~test_mask)
test_idx = np.flatnonzero(test_mask)

if y.iloc[train_idx].nunique() != 2 or y.iloc[test_idx].nunique() != 2:
    raise ValueError("Client holdout does not contain both target classes; do not silently switch to a row split.")

train_clients = set(clients.iloc[train_idx])
test_clients_actual = set(clients.iloc[test_idx])
assert train_clients.isdisjoint(test_clients_actual)

print("Split strategy: client_holdout")
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients_actual))
print("Client overlap:", len(train_clients & test_clients_actual))
print("Train declining rate:", f"{y.iloc[train_idx].mean():.3%}")
print("Test declining rate:", f"{y.iloc[test_idx].mean():.3%}")

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


Split strategy: client_holdout
Train rows: 27675
Test rows: 2325
Train clients: 26
Test clients: 6
Client overlap: 0
Train declining rate: 55.476%
Test declining rate: 39.097%


## 3. Train + compare vs my baseline

The Week-4 baseline is the deterministic `baseline_refresh_score`. I evaluate that score on the **same held-out test rows** used for the models. The primary ranking metric is `Precision@50`, because the practical question is whether the top small review queue contains more declining pages. Average precision and ROC AUC are secondary ranking diagnostics; precision, recall and F1 describe the 0.5 classification threshold.

In [14]:
def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    top = temp.sort_values("score", ascending=False).head(min(k, len(temp)))
    return float(top["y"].mean()) if len(top) else 0.0

def metric_row(name, y_true, scores):
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= 0.5).astype(int)
    return {
        "method": name,
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "average_precision": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "accuracy": accuracy_score(y_true, pred),
    }

# Reproduce the repository's Week-4 baseline score if the generated CSV is absent.
if BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
else:
    baseline = frame[["content_id", "client_id", "impressions_90d", "days_since_last_update",
                      "avg_position", "word_count", "ctr", "engagement_rate", "scroll_rate"]].copy()
    def percentile_rank(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)
    def normalize(s):
        s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
        lo, hi = s.min(), s.max()
        return pd.Series(np.zeros(len(s)), index=s.index) if hi == lo else (s - lo) / (hi - lo)
    baseline["visibility_score"] = percentile_rank(np.log1p(baseline["impressions_90d"]))
    baseline["freshness_risk_score"] = percentile_rank(baseline["days_since_last_update"])
    baseline["position_opportunity_score"] = (
        (1 - normalize(baseline["avg_position"].clip(lower=1, upper=50)))
        * baseline["visibility_score"] * (baseline["avg_position"] > 0).astype(int)
    )
    baseline["depth_gap_score"] = (1 - percentile_rank(baseline["word_count"])) * baseline["visibility_score"]
    baseline["baseline_refresh_score"] = (
        0.40 * baseline["visibility_score"]
        + 0.30 * baseline["freshness_risk_score"]
        + 0.25 * baseline["position_opportunity_score"]
        + 0.05 * baseline["depth_gap_score"]
    ).clip(0, 1)

baseline_lookup = baseline.set_index("content_id")["baseline_refresh_score"]
baseline_test_scores = frame.iloc[test_idx]["content_id"].map(baseline_lookup).fillna(0).to_numpy()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

models = {
    "logistic_regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE)),
    ]),
    "random_forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
            n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
        )),
    ]),
}

rows = [metric_row("week4_baseline", y_test, baseline_test_scores)]
model_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    model_scores[name] = scores
    rows.append(metric_row(name, y_test, scores))

results = pd.DataFrame(rows).sort_values("precision_at_50", ascending=False).reset_index(drop=True)
results["delta_vs_baseline_p50"] = results["precision_at_50"] - results.loc[results["method"] == "week4_baseline", "precision_at_50"].iloc[0]

display(results.round(4))

best_model_name = results.loc[results["method"] != "week4_baseline", "method"].iloc[0]
print("Selected model by Precision@50:", best_model_name)


,method,precision_at_50,average_precision,roc_auc,precision,recall,f1,accuracy,delta_vs_baseline_p50
0,random_forest,0.74,0.6182,0.7500,0.5610,0.7437,0.6395,0.6723,0.50
1,decision_tree,0.58,0.5753,0.7415,0.5686,0.7162,0.6339,0.6766,0.34
2,logistic_regression,0.40,0.5248,0.7037,0.5701,0.5589,0.5644,0.6628,0.16
3,week4_baseline,0.24,0.4676,0.6269,0.4986,0.1892,0.2743,0.6086,0.00


Selected model by Precision@50: random_forest


## 4. Errors and interpretation

I will inspect false positives and false negatives rather than treating one metric as the whole story. I will also use permutation importance on the held-out test set for the selected model. These are directional interpretations: importance shows which inputs the fitted model relied on for this split, not causal effects.

In [15]:
best_model = models[best_model_name]
best_scores = model_scores[best_model_name]
pred = (best_scores >= 0.5).astype(int)

error_frame = frame.iloc[test_idx][["content_id", "client_id", "content_type", "main_intent",
                                    "content_age_days", "days_since_last_update", "impressions_90d",
                                    "avg_position", "word_count"]].copy()
error_frame["actual"] = y_test.to_numpy()
error_frame["predicted"] = pred
error_frame["probability"] = best_scores
error_frame["error_type"] = np.select(
    [
        (error_frame["actual"] == 1) & (error_frame["predicted"] == 0),
        (error_frame["actual"] == 0) & (error_frame["predicted"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)

print("Error counts:")
print(error_frame["error_type"].value_counts())

print("\nFalse positives — highest predicted probability:")
display(error_frame.query("error_type == 'false_positive'").sort_values("probability", ascending=False).head(10))

print("False negatives — lowest predicted probability among actual declines:")
display(error_frame.query("error_type == 'false_negative'").sort_values("probability").head(10))

# Error rates by content type; suppress tiny groups.
error_by_type = (
    error_frame.groupby("content_type")
    .agg(n=("actual", "size"), actual_decline_rate=("actual", "mean"), predicted_decline_rate=("predicted", "mean"))
    .query("n >= 20")
    .sort_values("actual_decline_rate", ascending=False)
)
display(error_by_type.round(4))

# Permutation importance on the held-out test set.
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance.head(12).round(5))

baseline_p50 = results.loc[results["method"] == "week4_baseline", "precision_at_50"].iloc[0]
best_p50 = results.loc[results["method"] == best_model_name, "precision_at_50"].iloc[0]
delta = best_p50 - baseline_p50

print("\nInterpretation")
if delta > 0:
    print(f"The selected model measured {best_p50:.3f} Precision@50 versus {baseline_p50:.3f} for the Week-4 baseline, a directional improvement of {delta:+.3f} on this holdout.")
else:
    print(f"The selected model measured {best_p50:.3f} Precision@50 versus {baseline_p50:.3f} for the Week-4 baseline, so it did not beat the baseline on this holdout ({delta:+.3f}).")
print("False positives are pages the model prioritized even though the observed label was not declining; false negatives are declining pages the model failed to prioritize.")
print("Permutation importance is interpreted as model reliance on this test split, not as causal evidence.")


Error counts:
error_type
correct           1563
false_positive     529
false_negative     233
Name: count, dtype: int64

False positives — highest predicted probability:


,content_id,client_id,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,avg_position,word_count,actual,predicted,probability,error_type
23250,content_d2dffcc697a4,client_f74efabef1,keyword article,transactional,144,20,5091,14.1,4496.0,0,1,0.737130,false_positive
23559,content_00603b0349b4,client_f74efabef1,keyword article,informational,125,20,1076,25.6,2439.0,0,1,0.734944,false_positive
25913,content_331182ca4cae,client_f74efabef1,keyword article,informational,134,20,3026,35.9,3546.0,0,1,0.733631,false_positive
23750,content_e55b8ab078b0,client_f74efabef1,keyword article,informational,112,20,369,21.8,2192.0,0,1,0.733059,false_positive
10155,content_643f585dc7f7,client_f74efabef1,keyword article,transactional,104,20,761,25.1,1980.0,0,1,0.731120,false_positive
5966,content_f5013794ba57,client_f74efabef1,keyword article,transactional,175,20,881,15.7,3622.0,0,1,0.730532,false_positive
28337,content_ea4417d89e2c,client_f74efabef1,keyword article,transactional,112,20,352,11.9,2556.0,0,1,0.729056,false_positive
4249,content_db1cd41b4b4f,client_f74efabef1,keyword article,transactional,105,105,1482,12.9,2221.0,0,1,0.729052,false_positive
21530,content_b15a8dbdf66f,client_f74efabef1,keyword article,informational,144,20,1647,22.4,4095.0,0,1,0.727853,false_positive
2380,content_96da95476e63,client_f74efabef1,keyword article,informational,125,20,784,7.4,2598.0,0,1,0.724807,false_positive


False negatives — lowest predicted probability among actual declines:


,content_id,client_id,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,avg_position,word_count,actual,predicted,probability,error_type
5770,content_28b4223f4e5f,client_98a3ab7c34,keyword article,informational,91,1,1,0.0,3109.0,1,0,0.079867,false_negative
3879,content_34b14c00f80c,client_d4735e3a26,feedly article,unknown,308,20,3,0.0,659.0,1,0,0.082196,false_negative
27177,content_79ac977c6e0b,client_f74efabef1,keyword article,informational,104,8,3,0.7,2304.0,1,0,0.149546,false_negative
22991,content_472ce7ae14c0,client_d4735e3a26,feedly article,unknown,300,20,3,0.3,684.0,1,0,0.152184,false_negative
5608,content_a55d958ec725,client_d4735e3a26,feedly article,unknown,290,20,3,2.7,837.0,1,0,0.163987,false_negative
12864,content_f1ef151d5e36,client_d4735e3a26,feedly article,unknown,294,20,3,2.0,978.0,1,0,0.165946,false_negative
12076,content_230de4c50860,client_d4735e3a26,feedly article,unknown,288,20,3,2.0,824.0,1,0,0.169816,false_negative
25838,content_cbc3b52a2ac1,client_98a3ab7c34,keyword article,informational,125,1,2,3.0,2286.0,1,0,0.171345,false_negative
13659,content_4c437dd8c1ee,client_d4735e3a26,feedly article,unknown,284,20,3,3.0,840.0,1,0,0.174066,false_negative
23810,content_37804210415c,client_d4735e3a26,feedly article,unknown,300,20,4,2.0,813.0,1,0,0.174828,false_negative


,n,actual_decline_rate,predicted_decline_rate
content_type,,,
keyword article,1367,0.5238,0.7440
feedly article,958,0.2015,0.1962


,feature,importance_mean,importance_std
9,days_with_impressions,0.08784,0.00704
5,log_impressions_90d,0.03121,0.00386
13,ctr,0.02281,0.00701
16,scroll_rate,0.00901,0.00443
0,search_volume,0.00828,0.00121
6,log_clicks_90d,0.00825,0.00189
14,avg_position,0.00761,0.00493
20,main_intent,0.00399,0.00045
10,days_with_sessions,0.00302,0.00104
25,position_tier,0.00300,0.00344



Interpretation
The selected model measured 0.740 Precision@50 versus 0.240 for the Week-4 baseline, a directional improvement of +0.500 on this holdout.
False positives are pages the model prioritized even though the observed label was not declining; false negatives are declining pages the model failed to prioritize.
Permutation importance is interpreted as model reliance on this test split, not as causal evidence.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.